# Laboratorio de Práctica: ResNet y Transfer Learning

**Curso:** Redes Neuronales y Aprendizaje Profundo — UNI, FIIS, Unidad de Posgrado
**Clase:** 6 — ResNet ([He et al., 2016](https://arxiv.org/abs/1512.03385)) y Transfer Learning (CS231n, Lecture 9)

## Objetivos

Al terminar este laboratorio usted podrá:

1. Implementar el bloque residual básico y verificar experimentalmente el **problema de la degradación**:
   una red "plana" profunda entrena peor que una superficial, mientras que su versión residual no.
2. Implementar el bloque **bottleneck** ($1\times1\to3\times3\to1\times1$) y verificar su ahorro de cómputo
   frente a una alternativa sin reducción de dimensión.
3. Construir ResNet-18/34/50 completas siguiendo la Tabla 1 del artículo, y verificar su conteo de parámetros
   contra `torchvision`.
4. Aplicar **transfer learning** con un ResNet preentrenado en ImageNet, comparando las dos estrategias
   (extractor de características fijo vs.\ ajuste fino) en un dataset pequeño.

**Formato:** las celdas marcadas con `# TODO` son las que usted debe completar. El resto es código de apoyo
ya funcional. Ejecute las celdas en orden. Se recomienda usar un entorno con GPU
(Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU*).

---


In [ ]:
# Configuración inicial
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
import copy
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"PyTorch: {torch.__version__} | torchvision: {torchvision.__version__}")

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


## Parte A — El problema de la degradación

### A.1 Redes "planas" de distinta profundidad

Antes de tocar ResNet, reproduzcamos (a pequeña escala, en CIFAR-10) el fenómeno que motiva todo el artículo:
una red convolucional "plana" (una simple pila de capas 3x3, sin atajos) de más capas puede entrenar **peor**
que una versión más superficial — y esto no es sobreajuste, ya que el error de *entrenamiento* también empeora.

**Ejercicio A.1.** Complete `PlainNet`, una red totalmente convolucional (sin atajos) parametrizada por
`num_blocks_per_stage`, siguiendo el estilo de la Figura 3 (centro) del artículo: 3 etapas con 16, 32 y 64
canales respectivamente, cada etapa con `num_blocks_per_stage` bloques de 2 capas conv 3x3, y downsampling
(stride 2) al inicio de las etapas 2 y 3.


In [ ]:
class PlainBlock(nn.Module):
    """Bloque de 2 capas conv 3x3 SIN atajo (para el estudio del problema de degradación)."""

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))   # sin atajo: no se suma x
        return out


class PlainNet(nn.Module):
    """Red plana para CIFAR-10, estilo Fig. 3 (centro) del artículo: 3 etapas de canales {16,32,64}."""

    def __init__(self, num_blocks_per_stage: int, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        # TODO: construya las 3 etapas con nn.Sequential, usando PlainBlock.
        # - stage1: num_blocks_per_stage bloques, 16 -> 16 canales, stride 1 en todos
        # - stage2: num_blocks_per_stage bloques, 16 -> 32 canales, stride 2 en el PRIMER bloque, 1 en el resto
        # - stage3: num_blocks_per_stage bloques, 32 -> 64 canales, stride 2 en el PRIMER bloque, 1 en el resto
        self.stage1 = None
        self.stage2 = None
        self.stage3 = None

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    @staticmethod
    def _make_stage(in_channels, out_channels, num_blocks, first_stride):
        layers = [PlainBlock(in_channels, out_channels, stride=first_stride)]
        for _ in range(num_blocks - 1):
            layers.append(PlainBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


**Nota de implementación (para completar el TODO):** use `self._make_stage(...)` con los argumentos
correctos, por ejemplo: `self.stage1 = self._make_stage(16, 16, num_blocks_per_stage, first_stride=1)`.

Con `num_blocks_per_stage = n`, esta red tiene $6n+2$ capas con pesos (siguiendo la notación de la Sección 4.2
del artículo, que usa exactamente este esquema para sus experimentos con CIFAR-10). Verifique:


In [ ]:
for n in [3, 9]:
    net = PlainNet(num_blocks_per_stage=n)
    total_layers = 6 * n + 2
    print(f"n={n}: red plana de {total_layers} capas con pesos, {count_params(net)/1e3:.1f} K parámetros")


### A.2 La misma red, con atajos residuales

**Ejercicio A.2.** Complete `ResidualBlock`, idéntico a `PlainBlock` pero con la conexión de atajo identidad
(Ecuación 1 del artículo: $y = F(x,\{W_i\}) + x$). Cuando `stride != 1` o cambian los canales, use un atajo de
**proyección** ($1\times1$ conv, opción B del artículo) para igualar dimensiones.


In [ ]:
class ResidualBlock(nn.Module):
    """Bloque residual básico de 2 capas (Fig. 2 / Fig. 5-izquierda del artículo)."""

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Atajo: identidad si las dimensiones coinciden; si no, proyección 1x1 (opción B del artículo).
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        # TODO: implemente F(x) = conv1->bn1->relu->conv2->bn2 (SIN relu al final de esta parte),
        # sume el atajo self.shortcut(x), y aplique la ReLU final DESPUÉS de la suma
        # (ver Fig. 2 del artículo: "we adopt the second nonlinearity after the addition").
        out = None
        return out


class ResNetCifar(nn.Module):
    """Misma arquitectura que PlainNet pero con ResidualBlock (Sección 4.2 del artículo)."""

    def __init__(self, num_blocks_per_stage: int, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.stage1 = self._make_stage(16, 16, num_blocks_per_stage, first_stride=1)
        self.stage2 = self._make_stage(16, 32, num_blocks_per_stage, first_stride=2)
        self.stage3 = self._make_stage(32, 64, num_blocks_per_stage, first_stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)

    @staticmethod
    def _make_stage(in_channels, out_channels, num_blocks, first_stride):
        layers = [ResidualBlock(in_channels, out_channels, stride=first_stride)]
        for _ in range(num_blocks - 1):
            layers.append(ResidualBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


**Ejercicio A.3 (verificación conceptual).** El artículo señala que la conexión de atajo identidad
**no añade parámetros ni cómputo adicional** salvo la suma elemento a elemento. Verifique esto comparando
`PlainNet` y `ResNetCifar` con el mismo `num_blocks_per_stage`: deben tener casi el mismo número de parámetros
(la única diferencia son las proyecciones $1\times1$ en los 2 puntos donde cambian canales/resolución).


In [ ]:
n = 3
plain = PlainNet(num_blocks_per_stage=n)
resnet = ResNetCifar(num_blocks_per_stage=n)

p_plain = count_params(plain)
p_resnet = count_params(resnet)
print(f"PlainNet   (n={n}): {p_plain:,} parámetros")
print(f"ResNetCifar(n={n}): {p_resnet:,} parámetros")
print(f"Diferencia: {p_resnet - p_plain:,} parámetros (proyecciones 1x1 de las 2 transiciones de etapa)")


### A.4 Reproducir (a pequeña escala) el problema de la degradación

**Ejercicio A.4.** Entrene `PlainNet` y `ResNetCifar` con $n=3$ (20 capas) y $n=9$ (56 capas) sobre CIFAR-10
durante pocas épocas, y compare las curvas de error de **entrenamiento**. Con suficientes épocas debería
observar el mismo patrón cualitativo que la Figura 1 del artículo: la red plana de 56 capas entrena peor que
la de 20 capas, mientras que ambas versiones residuales entrenan razonablemente bien (o la de 56 capas
incluso mejor). Con pocas épocas en un laboratorio de clase, es posible que sólo se insinúe la tendencia —
lo importante es entender el experimento y el criterio de comparación.


In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

trainset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)


In [ ]:
def train_and_track(model, trainloader, epochs=5, lr=0.1):
    """Entrena y devuelve el historial de error de ENTRENAMIENTO por época (lo relevante para
    ilustrar el problema de la degradación, que se manifiesta como mayor error de entrenamiento)."""
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    # Como en el artículo (Sección 4.2): SGD, momento 0.9, weight decay 1e-4
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)

    train_err_history = []
    for epoch in range(epochs):
        model.train()
        correct, total = 0, 0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)

            # TODO: paso de entrenamiento estándar (zero_grad, forward, loss, backward, step)
            optimizer.zero_grad()
            outputs = None       # <-- complete
            loss = None          # <-- complete (criterion(outputs, labels))
            # backward y step
            pass

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_err = 100.0 * (1 - correct / total)
        train_err_history.append(train_err)
        print(f"  Época {epoch+1}/{epochs} | error de entrenamiento: {train_err:.2f}%")

    return train_err_history


EPOCHS = 5   # aumente este valor si dispone de más tiempo de GPU

print("=== PlainNet, n=3 (20 capas) ===")
err_plain20 = train_and_track(PlainNet(3), trainloader, epochs=EPOCHS)

print("\n=== PlainNet, n=9 (56 capas) ===")
err_plain56 = train_and_track(PlainNet(9), trainloader, epochs=EPOCHS)

print("\n=== ResNetCifar, n=3 (20 capas) ===")
err_res20 = train_and_track(ResNetCifar(3), trainloader, epochs=EPOCHS)

print("\n=== ResNetCifar, n=9 (56 capas) ===")
err_res56 = train_and_track(ResNetCifar(9), trainloader, epochs=EPOCHS)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(err_plain20, label="plain-20", marker='o')
axes[0].plot(err_plain56, label="plain-56", marker='o')
axes[0].set_title("Redes planas")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("Error de entrenamiento (%)")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(err_res20, label="ResNet-20", marker='o')
axes[1].plot(err_res56, label="ResNet-56", marker='o')
axes[1].set_title("ResNets")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Error de entrenamiento (%)")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


**Pregunta A.4.** Compare el error de entrenamiento final de plain-56 contra plain-20, y de ResNet-56 contra
ResNet-20. ¿Observa (aunque sea de forma incipiente, dado el número reducido de épocas) el patrón de la Figura 1
del artículo? ¿Por qué comparamos el error de **entrenamiento** y no sólo el de prueba para diagnosticar el
problema de la degradación?


*(Responda aquí)*

---
## Parte B — El bloque *bottleneck*

### B.1 Implementación

**Ejercicio B.1.** Complete `BottleneckBlock` (Figura 5-derecha del artículo): 3 capas
conv $1\times1$ (reduce a `mid_channels`) $\to$ conv $3\times3$ (opera en `mid_channels`) $\to$
conv $1\times1$ (expande a `out_channels`), con atajo identidad o de proyección según corresponda.


In [ ]:
class BottleneckBlock(nn.Module):
    """Bloque bottleneck de 3 capas (Fig. 5-derecha), usado en ResNet-50/101/152."""
    expansion = 4   # out_channels = mid_channels * expansion, como en la Tabla 1 del artículo

    def __init__(self, in_channels, mid_channels, stride=1):
        super().__init__()
        out_channels = mid_channels * self.expansion

        # TODO: conv1 = conv1x1 (in_channels -> mid_channels) + BN
        self.conv1 = None
        self.bn1 = None

        # TODO: conv2 = conv3x3 (mid_channels -> mid_channels, stride=stride, padding=1) + BN
        self.conv2 = None
        self.bn2 = None

        # TODO: conv3 = conv1x1 (mid_channels -> out_channels) + BN
        self.conv3 = None
        self.bn3 = None

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))          # sin ReLU todavía
        out = out + self.shortcut(x)
        return F.relu(out)


### B.2 Verificación contra el ejemplo de la Figura 5 del artículo

La Figura 5 (derecha) del artículo muestra un ejemplo concreto: entrada de 256 canales, reducción a 64 canales
vía conv $1\times1$, conv $3\times3$ sobre 64 canales, y expansión de vuelta a 256 vía conv $1\times1$.

**Ejercicio B.2.** Instancie `BottleneckBlock(in_channels=256, mid_channels=64, stride=1)` y verifique que:
(a) la forma de salida es la misma que la de entrada, y (b) el atajo usado es la identidad (sin parámetros),
ya que `in_channels == out_channels` y `stride == 1`.


In [ ]:
block = BottleneckBlock(in_channels=256, mid_channels=64, stride=1)

x = torch.randn(2, 256, 28, 28)
y = block(x)
print(f"Forma de entrada:  {tuple(x.shape)}")
print(f"Forma de salida:   {tuple(y.shape)}   (esperado: igual a la entrada)")
assert y.shape == x.shape

is_identity_shortcut = isinstance(block.shortcut, nn.Sequential) and len(block.shortcut) == 0
print(f"¿Atajo identidad (sin parámetros)?  {is_identity_shortcut}")
assert is_identity_shortcut, "Con in_channels==out_channels y stride=1 el atajo debería ser identidad."


**Ejercicio B.3 (el argumento clave del artículo sobre el atajo identidad).** El artículo argumenta que si el
atajo identidad de un bloque *bottleneck* se reemplazara por una proyección, el costo computacional y el tamaño
del modelo se **duplicarían**, porque el atajo conectaría los extremos de alta dimensión (256-d) en vez de los de
baja dimensión (64-d). Calcule usted mismo ambos casos para el bloque de la celda anterior.


In [ ]:
def shortcut_projection_params(in_channels, out_channels):
    """Parámetros (sin sesgos) de una proyección 1x1 in_channels -> out_channels."""
    return in_channels * out_channels

# TODO: calcule los parámetros de una proyección conectando los extremos ALTOS (256 -> 256)
params_high_dim_shortcut = None   # shortcut_projection_params(256, 256)

# Como referencia, el bloque ya usa una identidad (0 parámetros) porque in==out channels;
# pero imaginemos el caso hipotético en que SÍ hiciera falta una proyección para cambiar dimensión,
# por ejemplo al pasar de 256 a 512 canales en la siguiente etapa:
# TODO: calcule los parámetros de una proyección conectando 256 (bajo, post-reducción) -> algo pequeño, p.ej. 64->128
params_low_dim_example = None    # shortcut_projection_params(64, 128)

print(f"Proyección sobre los extremos de ALTA dimensión (256->256): {params_high_dim_shortcut:,} parámetros")
print(f"Proyección sobre dimensión reducida (64->128, sólo ilustrativo): {params_low_dim_example:,} parámetros")
print("\nConclusión: conectar el atajo a los extremos de alta dimensión es mucho más costoso;")
print("por eso el atajo identidad (0 parámetros) es especialmente valioso en el diseño bottleneck.")


---
## Parte C — ResNet completa para ImageNet

### C.1 Ensamblando ResNet-18/34/50 siguiendo la Tabla 1 del artículo

**Ejercicio C.1.** Complete `make_resnet_stage` y las listas `RESNET_CONFIGS`, siguiendo exactamente la
Tabla 1 del artículo (también reproducida como Cuadro 1 en la clase): número de bloques por etapa para
ResNet-18, 34 y 50, y qué tipo de bloque (básico o *bottleneck*) usa cada uno.


In [ ]:
RESNET_CONFIGS = {
    # (tipo_de_bloque, [núm. de bloques en conv2_x, conv3_x, conv4_x, conv5_x])
    'resnet18': (ResidualBlock,   [2, 2, 2, 2]),
    'resnet34': (ResidualBlock,   [3, 4, 6, 3]),
    # TODO: complete resnet50 usando BottleneckBlock y los números de bloques de la Tabla 1
    # (conv2_x: 3, conv3_x: 4, conv4_x: 6, conv5_x: 3 -- los mismos conteos que ResNet-34,
    #  pero con bloques bottleneck de 3 capas en vez de básicos de 2)
    'resnet50': (None, []),
}


class ResNetImageNet(nn.Module):
    """ResNet genérica para ImageNet (224x224), siguiendo la Tabla 1 del artículo."""

    def __init__(self, block_type, num_blocks_list, num_classes=1000):
        super().__init__()
        self.in_channels = 64

        # Stem: conv 7x7/2 + maxpool 3x3/2 (igual para las 3 configuraciones)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # Determinar "mid_channels"/"out_channels" base de cada etapa según la Tabla 1
        base_channels = [64, 128, 256, 512]
        strides = [1, 2, 2, 2]   # conv2_x no reduce resolución (ya la redujo el stem); las demás sí

        stages = []
        for out_ch, num_blocks, stride in zip(base_channels, num_blocks_list, strides):
            stages.append(self._make_stage(block_type, out_ch, num_blocks, stride))
        self.stages = nn.Sequential(*stages)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        final_channels = base_channels[-1] * getattr(block_type, 'expansion', 1)
        self.fc = nn.Linear(final_channels, num_classes)

    def _make_stage(self, block_type, out_channels, num_blocks, first_stride):
        layers = []
        expansion = getattr(block_type, 'expansion', 1)
        if block_type is ResidualBlock:
            layers.append(block_type(self.in_channels, out_channels, stride=first_stride))
            self.in_channels = out_channels
            for _ in range(num_blocks - 1):
                layers.append(block_type(self.in_channels, out_channels, stride=1))
        else:  # BottleneckBlock: el 2do argumento es mid_channels, no out_channels
            layers.append(block_type(self.in_channels, out_channels, stride=first_stride))
            self.in_channels = out_channels * expansion
            for _ in range(num_blocks - 1):
                layers.append(block_type(self.in_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.stages(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


def build_resnet(name, num_classes=1000):
    block_type, num_blocks_list = RESNET_CONFIGS[name]
    return ResNetImageNet(block_type, num_blocks_list, num_classes=num_classes)


### C.2 Verificación de forma y comparación con torchvision

**Ejercicio C.2.** Construya `resnet18`, `resnet34` y (una vez complete el TODO de C.1) `resnet50`, verifique
que un tensor $224\times224$ produce logits `(batch, 1000)`, y compare el conteo de parámetros contra
`torchvision.models`. Los valores deberían ser muy cercanos (pequeñas diferencias por cómo se cuentan sesgos
o la capa stem).


In [ ]:
from torchvision.models import resnet18, resnet34, resnet50

x = torch.randn(2, 3, 224, 224)

for name, tv_ctor in [('resnet18', resnet18), ('resnet34', resnet34), ('resnet50', resnet50)]:
    model = build_resnet(name, num_classes=1000)
    with torch.no_grad():
        out = model(x)
    tv_model = tv_ctor(weights=None)

    print(f"{name}:")
    print(f"  forma de salida: {tuple(out.shape)}  (esperado: (2, 1000))")
    print(f"  parámetros (nuestra implementación): {count_params(model)/1e6:.2f} M")
    print(f"  parámetros (torchvision):            {count_params(tv_model)/1e6:.2f} M")
    print()


**Pregunta C.2.** Según la Tabla 1 del artículo, ResNet-34 tiene 3.6 mil millones de FLOPs, sólo el 18% de
los 19.6 mil millones de VGG-19. ¿Por qué una red con *más capas* que VGG-16 (34 vs. 16) puede tener *menos*
cómputo total? (Piense en el ancho de los canales y en el uso de bottlenecks/campos receptivos pequeños en
todas las capas frente a las capas FC de VGG.)


*(Responda aquí)*

---
## Parte D — Transfer Learning con ResNet preentrenada

Usaremos un ResNet-18 preentrenado en ImageNet para clasificar un dataset pequeño y distinto: el clásico
dataset de **hormigas vs.\ abejas** (120 imágenes de entrenamiento y 75 de validación por clase — el mismo
usado en el tutorial oficial de PyTorch sobre transfer learning). Es un caso de **dataset pequeño y similar**
a ImageNet (son fotos naturales de insectos, un dominio cercano a las clases de ImageNet), por lo que, según la
matriz de decisión de la clase, ambas estrategias deberían funcionar razonablemente bien — lo interesante es
comparar su comportamiento.


In [ ]:
import urllib.request
import zipfile

data_dir = "./hymenoptera_data"
if not os.path.exists(data_dir):
    url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
    zip_path = "hymenoptera_data.zip"
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Dataset descargado y extraído.")
else:
    print("Dataset ya presente.")

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),  # estadísticas de ImageNet
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
}

image_datasets = {x: torchvision.datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
                   for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=16, shuffle=(x == 'train'), num_workers=2)
               for x in ['train', 'val']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print(f"Clases: {class_names}")
print(f"Tamaños: {dataset_sizes}")


### D.1 Estrategia 1 — Extractor de características fijo

**Ejercicio D.1.** Cargue un `resnet18` preentrenado, congele todos sus parámetros, y reemplace la capa `fc`
final por una nueva (no congelada) para las 2 clases del dataset.


In [ ]:
from torchvision.models import ResNet18_Weights

def build_feature_extractor(num_classes=2):
    model = torchvision.models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

    # TODO: congele TODOS los parámetros del modelo (requires_grad = False)
    for param in model.parameters():
        pass  # <-- complete

    # TODO: reemplace model.fc por una nueva capa nn.Linear(in_features, num_classes).
    # Pista: la capa original expone su tamaño de entrada en model.fc.in_features.
    # Una capa nn.Linear recién creada SIEMPRE tiene requires_grad=True por defecto,
    # así que sólo esta capa se entrenará.
    model.fc = None

    return model.to(device)

model_fixed = build_feature_extractor(num_classes=len(class_names))

trainable = sum(p.numel() for p in model_fixed.parameters() if p.requires_grad)
total = count_params(model_fixed)
print(f"Parámetros entrenables: {trainable:,} de {total:,} totales ({100*trainable/total:.2f}%)")


### D.2 Estrategia 2 — Ajuste fino completo

**Ejercicio D.2.** Cargue otro `resnet18` preentrenado, reemplace `fc` de la misma forma, pero esta vez
**sin** congelar el resto de la red (todos los parámetros permanecen entrenables).


In [ ]:
def build_finetune_model(num_classes=2):
    model = torchvision.models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    # TODO: reemplace model.fc igual que en D.1 (aquí NO se congela nada más)
    model.fc = None
    return model.to(device)

model_finetune = build_finetune_model(num_classes=len(class_names))
print(f"Parámetros entrenables: {sum(p.numel() for p in model_finetune.parameters() if p.requires_grad):,} de {count_params(model_finetune):,} totales")


### D.3 Entrenamiento y comparación

**Ejercicio D.3.** Complete el bucle de entrenamiento/validación y ejecútelo para ambos modelos.
Note las diferencias deliberadas en la configuración del optimizador:
- Extractor fijo: se puede usar una tasa de aprendizaje "normal" (p.ej. 0.01) ya que sólo se entrena una capa nueva.
- Ajuste fino: se usa una tasa de aprendizaje mucho **menor** (p.ej. 0.001), como se explicó en la clase, para no
  destruir las características preentrenadas.


In [ ]:
def train_transfer_model(model, dataloaders, dataset_sizes, epochs=10, lr=0.01):
    criterion = nn.CrossEntropyLoss()
    # Sólo se pasan al optimizador los parámetros con requires_grad=True
    optimizer = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, momentum=0.9)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()

            running_loss, running_corrects = 0.0, 0
            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    # TODO: forward pass y cálculo de la pérdida
                    outputs = None        # <-- complete: model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = None            # <-- complete: criterion(outputs, labels)

                    if phase == 'train':
                        # TODO: backward y step
                        pass

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = (running_corrects.double() / dataset_sizes[phase]).item()
            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc)

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_wts = copy.deepcopy(model.state_dict())

        print(f"Época {epoch+1}/{epochs} | train acc: {history['train_acc'][-1]:.3f} | val acc: {history['val_acc'][-1]:.3f}")

    model.load_state_dict(best_wts)
    print(f"\nMejor exactitud de validación: {best_acc:.3f}")
    return model, history


print("=== Estrategia 1: extractor de características fijo ===")
model_fixed, hist_fixed = train_transfer_model(model_fixed, dataloaders, dataset_sizes, epochs=10, lr=0.01)

print("\n=== Estrategia 2: ajuste fino completo ===")
model_finetune, hist_finetune = train_transfer_model(model_finetune, dataloaders, dataset_sizes, epochs=10, lr=0.001)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(hist_fixed["train_acc"], label="extractor fijo (train)", linestyle='--')
axes[0].plot(hist_fixed["val_acc"], label="extractor fijo (val)")
axes[0].plot(hist_finetune["train_acc"], label="ajuste fino (train)", linestyle='--')
axes[0].plot(hist_finetune["val_acc"], label="ajuste fino (val)")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("Exactitud"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[0].set_title("Exactitud")

axes[1].plot(hist_fixed["train_loss"], label="extractor fijo (train)", linestyle='--')
axes[1].plot(hist_fixed["val_loss"], label="extractor fijo (val)")
axes[1].plot(hist_finetune["train_loss"], label="ajuste fino (train)", linestyle='--')
axes[1].plot(hist_finetune["val_loss"], label="ajuste fino (val)")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Pérdida"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
axes[1].set_title("Pérdida")

plt.tight_layout()
plt.show()

print(f"\n{'Estrategia':<25}{'Mejor exactitud val.':>22}{'Parámetros entrenados':>24}")
print(f"{'Extractor fijo':<25}{max(hist_fixed['val_acc']):22.3f}{sum(p.numel() for p in model_fixed.parameters() if p.requires_grad):24,}")
print(f"{'Ajuste fino':<25}{max(hist_finetune['val_acc']):22.3f}{sum(p.numel() for p in model_finetune.parameters() if p.requires_grad):24,}")


### D.4 Visualizar algunas predicciones

**Ejercicio D.4 (opcional).** Visualice algunas imágenes de validación junto con la predicción del modelo de
ajuste fino, para inspeccionar cualitativamente los resultados.


In [ ]:
def imshow(inp, title=None):
    inp = inp.numpy().transpose((1, 2, 0))
    mean = [0.485, 0.456, 0.406]; std = [0.229, 0.224, 0.225]
    inp = std * inp + mean
    inp = inp.clip(0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.axis('off')


model_finetune.eval()
inputs, labels = next(iter(dataloaders['val']))
inputs, labels = inputs.to(device), labels.to(device)
with torch.no_grad():
    outputs = model_finetune(inputs)
    _, preds = torch.max(outputs, 1)

fig = plt.figure(figsize=(12, 6))
for i in range(min(6, inputs.size(0))):
    ax = plt.subplot(2, 3, i + 1)
    correct = "✓" if preds[i] == labels[i] else "✗"
    imshow(inputs.cpu().data[i], title=f"{correct} pred: {class_names[preds[i]]}")
plt.tight_layout()
plt.show()


---
## Parte E — Preguntas de cierre (entregable)

Responda de forma breve (3-5 líneas cada una) en las celdas de texto provistas.

**E.1.** Con base en sus resultados de la Parte D, ¿qué estrategia (extractor fijo o ajuste fino) obtuvo mejor
exactitud de validación en este dataset? ¿Coincide con lo que predeciría la matriz de decisión de la clase,
dado que hymenoptera_data es un dataset **pequeño** y **similar** a ImageNet?


*(Responda aquí)*

**E.2.** En la Parte A, entrenamos con pocas épocas por restricciones de tiempo de laboratorio. Si tuviera
acceso a mucho más cómputo, ¿qué esperaría que pasara con la brecha entre plain-20 y plain-56 al entrenar por
muchas más iteraciones? Relacione su respuesta con lo que el artículo reporta en la Figura 1 (curvas completas
de entrenamiento en ImageNet).


*(Responda aquí)*

**E.3.** En la Parte D usamos tasas de aprendizaje distintas para cada estrategia (0.01 para el extractor
fijo, 0.001 para el ajuste fino). Explique por qué esta diferencia es importante, y qué podría salir mal si
usara la misma tasa de aprendizaje alta para el ajuste fino completo.


*(Responda aquí)*

**E.4 (opcional, reto).** Modifique `RESNET_CONFIGS` para añadir `'resnet101'` (bottleneck, bloques
`[3, 4, 23, 3]` según la Tabla 1) y verifique que su conteo de parámetros se acerca al de
`torchvision.models.resnet101(weights=None)`.


In [ ]:
# Espacio para el reto E.4


---
### Referencias

- He, K., Zhang, X., Ren, S., & Sun, J. (2016). *Deep Residual Learning for Image Recognition*. CVPR. [arXiv:1512.03385](https://arxiv.org/abs/1512.03385)
- Li, F.-F., Johnson, J., & Yeung, S. (2017). *CS231n: Convolutional Neural Networks for Visual Recognition*, Lecture 9: CNN Architectures. Stanford University.
- PyTorch Tutorials: [Transfer Learning for Computer Vision Tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html) (fuente del dataset hymenoptera_data).
